In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HMAMP-main)

This notebook curates the **HMAMP-main** source by standardizing two types of inputs: a labeled hemolysis dataset and an additional set of sequences without label information. The pipeline cleans peptide sequences, applies duplicate consistency checks separately for labeled and unlabeled data, builds metadata, and exports curated outputs for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HMAMP-main
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:
- **Loads a labeled dataset** from `hemolythic.csv`:
  - renames `text` → `sequence` and `labels` → `label`,
  - removes whitespace from sequences,
  - keeps the standardized schema `(sequence, label)` with labels provided by the source.
- **Loads an unlabeled sequence list** from `hemo.txt`:
  - reads one sequence per line,
  - removes whitespace from sequences,
  - assigns a placeholder `label = 2` to explicitly mark these as **unlabeled/unknown**.
- **Checks duplicated sequences** separately for both datasets:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv` (deduplicated labeled dataset),
  - `detected_error_sequences.csv` (conflicting-label duplicates from the labeled set),
  - `metadata.json`.

In [2]:
name_source = "HMAMP-main"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_hemolythic = (
    pd.read_csv(f"{PATH_INPUT}/{name_source}/hemolythic.csv")
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(sequence=lambda df: df["sequence"]
        .astype(str).str.replace(" ", "", regex=False))
    [["sequence", "label"]]
)

In [4]:
df_hemo_unlabel = (
    pd.read_csv(f"{PATH_INPUT}/{name_source}/hemo.txt", names=["sequence"])
    .assign(sequence=lambda df: df["sequence"]
            .astype(str).str.replace(" ", "", regex=False))
    .assign(label=2) # There is no information about the labels of this source, therefore it will be identified with a 2
)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_hemolythic, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_remove_duplicated_unlabel, df_errors_unlabel, df_unique_unlabel = processing_duplicated(df_hemo_unlabel, group_seq="sequence", sort_key="label")
df_full_unlabel = pd.concat([df_unique_unlabel, df_remove_duplicated_unlabel], axis=0)

In [7]:
df_errors.shape

(0, 1)

In [8]:
df_errors_unlabel.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_hemolythic)),
    "number_of_sequences_retained": int(len(df_full) + len(df_full_unlabel)),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "number_of_unlabel_sequences" : len(df_full_unlabel),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 9, 3, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'No information;Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://github.com/wl-wl/HMAMP-main/tree/main/data',
 'publication': 'https://pubs.acs.org/doi/abs/10.1021/acs.jmedchem.4c03073',
 'number_of_raw_sequences': 1104,
 'number_of_sequences_retained': 1656,
 'number_of_positive_sequences': 552,
 'number_of_negative_sequences': 552,
 'number_of_erroneous_sequences': 0,
 'number_of_unlabel_sequences': 552,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_unlabel.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)